# Transfer Learning with EfficientNet-B0
Fine-tuning a pre-trained EfficientNet model on the Food-101 subset.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import torch
import torch.nn as nn
import torch.optim as optim
import mlflow
from src.food_classifier import FoodEfficientNet

## 1. Load Pretrained EfficientNet-B0

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Start with frozen backbone (only train classifier head)
model = FoodEfficientNet(num_classes=20, freeze_backbone=True).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.model.classifier.parameters(), lr=0.001)


## 2. MLflow Tracking: Phase 1 (Frozen Backbone)

In [ ]:
mlflow.set_experiment('Food101_EfficientNet_TransferLearning')

with mlflow.start_run(run_name='Phase_1_FrozenHead'):
    mlflow.log_param('learning_rate', 0.001)
    mlflow.log_param('epochs', 3)
    mlflow.log_param('architecture', 'EfficientNet-B0')
    mlflow.log_param('phase', 'training_head')

    # Mock forward pass
    dummy_inputs = torch.randn(16, 3, 224, 224).to(device)
    dummy_labels = torch.randint(0, 20, (16,)).to(device)
    
    model.train()
    optimizer.zero_grad()
    outputs = model(dummy_inputs)
    loss = criterion(outputs, dummy_labels)
    loss.backward()
    optimizer.step()
    
    mlflow.log_metric('train_loss', loss.item(), step=1)
    print(f'Phase 1 - Mock Loss: {loss.item():.4f}')


## 3. Phase 2: Unfreeze Top Layers for Fine-Tuning

In [ ]:
# Unfreeze top blocks
model.unfreeze_top_layers(blocks_to_unfreeze=3)

# Must create a new optimizer covering the newly unfrozen parameters,
# typically with a lower learning rate.
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

with mlflow.start_run(run_name='Phase_2_FineTuning'):
    mlflow.log_param('learning_rate', 1e-4)
    mlflow.log_param('epochs', 5)
    mlflow.log_param('phase', 'fine_tuning_top')

    model.train()
    optimizer.zero_grad()
    outputs = model(dummy_inputs)
    loss = criterion(outputs, dummy_labels)
    loss.backward()
    optimizer.step()
    
    mlflow.log_metric('train_loss', loss.item(), step=1)
    print(f'Phase 2 - Mock Loss: {loss.item():.4f}')


## 4. Save Best Model

In [ ]:
os.makedirs('../models', exist_ok=True)
torch.save(model.state_dict(), '../models/efficientnet_b0_food101.pth')
print("Model checkpoint saved successfully.")
